
# Wordle letter explorer

Which letters does an optimal strategy reach for **first**? This notebook ports the
hangman tier-list generator (the last section of
`hangman/math/notebooks/letter_analysis.ipynb`, "Greedily grow the set to minimise
outside-before-set pairs") to Wordle, over the committed optimal trees.

**The Wordle adaptation.** In hangman a guess *is* a letter, so a game hands you a
strict order over letters. In Wordle a guess is a **word**, so up to five letters
first appear at the same moment. A letter's *appearance time* on a path is therefore
the index of the first guess WORD containing it, and letters that enter on the same
guess are **tied — neither is before the other, so they are never inverted**.

Everything else follows hangman: the never-guessed penalty, uniform weighting, the
greedy accumulation, and `top2_diff` as the tier-boundary hint.

> **The objective these trees optimise is CONSTRAINED.** Each tree minimises the
> TOTAL guesses over the 2315 answers *subject to resolving every answer within 6*.
> That is not the same as minimum expected guesses, and any ranking derived here
> inherits that constraint.

In [ ]:

from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'wordle.py').exists():
            return c
    raise FileNotFoundError('Could not find wordle.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
for p in (math_dir, math_dir / 'tools'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import gzip
import numpy as np
import pandas as pd
from IPython.display import display
import paths                      # the canonical solver locations (math/tools/paths.py)

pd.set_option('display.max_rows', 200)   # show up to 200 rows before pandas truncates to a head/tail preview

print(f'imported paths from {math_dir}')
print(f'solver = {paths.SOLVER!r}')
print(f'trees  = {paths.TREES}')


## Defaults

`NUM_TREES` is the only knob: how many optimal trees to pool. The openers are ranked
by their **total guesses over the 2315 answers** (equivalently by `expected`, the same
ordering) from the committed `opener_totals.csv`, ties broken alphabetically, and the
best `NUM_TREES` are taken. `NUM_TREES = 1` is **SALET alone** (total 7920).

Every pooled path carries the same weight, so `NUM_TREES = 1` reduces exactly to the
single-tree case and larger values average the letter order across several equally
good strategies.

In [ ]:

NUM_TREES = 1     # <-- how many optimal trees to pool (top N openers by expected guesses; 1 = SALET)
MIN_GAP   = 0.5   # <-- greedy printout: blank line when top2_diff exceeds this (a tier-boundary hint)

print(f'pooling the top {NUM_TREES} tree(s); blank line in the greedy log at top2_diff > {MIN_GAP}')


## Loading the paths

Each tree file is one line per answer:

    <answer> <nguesses> <g1> <p1> <g2> <p2> ... <gN>

so **every line already is a path** — no tree walk needed. The guesses are `parts[2::2]`
(the same slicing `tools/checktrees.py` and `tools/depths.py` use), and the final guess
is always the answer itself, so the answer's own letters always appear.

For each path we record, per letter, the index of the **first guess word** containing it.
Letters are stored in first-appearance order; letters sharing a rank are tied. `occ` is
that letter's number of occurrences **in the answer** (0 for a letter typed but absent),
which is what "revealed" means for `set_before_outside` below.

The summed guess counts are checked against each opener's CSV total — the cheapest
end-to-end gate that the tree matches its reported value.

In [ ]:

SENTINEL = 26          # padding column value; letter indices are 0..25
BIG = 1 << 30          # padding rank: never earlier than anything real


def top_openers(num_trees):
    """The best `num_trees` openers by total guesses over the 2315 answers.

    Sourced from `paths.load_totals()` (the committed sweep CSV) -- never re-derived.
    `total` and `expected` induce the same ordering (expected = total / 2315); ties are
    broken alphabetically, so a strict top-N is taken."""
    totals = paths.load_totals()
    ranked = sorted(totals.items(), key=lambda kv: (kv[1], kv[0]))
    if num_trees > len(ranked):
        raise ValueError(f'only {len(ranked)} openers solved, asked for {num_trees}')
    return ranked[:num_trees]


def read_tree(opener):
    """Every path of one opener's optimal tree as (answer, [guess, ...])."""
    p = Path(paths.TREES) / f'{opener}.txt.gz'
    if not p.exists():
        raise FileNotFoundError(f'no committed tree for {opener!r} at {p}')
    out = []
    with gzip.open(p, 'rt') as f:
        for line in f:
            parts = line.split()
            out.append((parts[0], parts[2::2]))
    return out


def collect_paths(num_trees=None, verbose=True):
    """Pool the top `num_trees` trees' paths into padded arrays (built ONCE -- a path's
    letter order does not depend on the candidate set).

    Returns a dict:
      order (P, M) int   letter indices 0..25 in first-appearance order, padded with 26
      rank  (P, M) int   the index of the guess WORD the letter first appeared in
      block (P, M) int   column index where this column's rank-block starts (the tie group)
      occ   (P, M) int   occurrences of that letter in the ANSWER (0 if absent from it)
      valid (P, M) bool  real (non-padding) columns
      n     int          number of pooled paths
      openers            [(opener, total), ...] actually pooled
    """
    if num_trees is None:
        num_trees = NUM_TREES
    openers = top_openers(num_trees)

    recs = []
    for opener, total in openers:
        tree = read_tree(opener)
        assert len(tree) == paths.N_ANSWERS, f'{opener}: {len(tree)} paths, expected {paths.N_ANSWERS}'
        assert sum(len(g) for _a, g in tree) == total, f'{opener}: tree sum != CSV total {total}'
        for answer, guesses in tree:
            first = {}
            for gi, g in enumerate(guesses):
                for ch in g:
                    if ch not in first:
                        first[ch] = gi                 # the guess WORD it first appeared in
            letters = sorted(first, key=lambda c: (first[c], c))   # ties keep a stable order
            recs.append(([ord(c) - 97 for c in letters],
                         [first[c] for c in letters],
                         [answer.count(c) for c in letters]))

    P = len(recs)
    M = max(len(o) for o, _r, _c in recs)
    order = np.full((P, M), SENTINEL, dtype=np.int64)
    rank = np.full((P, M), BIG, dtype=np.int64)
    occ = np.zeros((P, M), dtype=np.int64)
    vlen = np.zeros(P, dtype=np.int64)
    for i, (o, r, c) in enumerate(recs):
        k = len(o)
        order[i, :k], rank[i, :k], occ[i, :k], vlen[i] = o, r, c, k

    col = np.arange(M)
    valid = col[None, :] < vlen[:, None]
    # block[i, k] = first column of the tie group (same rank) containing column k.
    # rank is non-decreasing along a row, so a running max of the group starts works.
    is_new = np.ones((P, M), dtype=bool)
    is_new[:, 1:] = rank[:, 1:] != rank[:, :-1]
    block = np.maximum.accumulate(np.where(is_new, col[None, :], 0), axis=1)

    if verbose:
        names = ', '.join(f'{w} ({t})' for w, t in openers[:6])
        more = '' if len(openers) <= 6 else f', ... (+{len(openers) - 6} more)'
        print(f'pooled {P:,} paths from {len(openers)} tree(s): {names}{more}')
        print(f'distinct letters per path: min {vlen.min()}, mean {vlen.mean():.2f}, max {vlen.max()}')
    return {'order': order, 'rank': rank, 'block': block, 'occ': occ,
            'valid': valid, 'n': P, 'openers': openers}


D = collect_paths()


## The two metrics

Given a set of letters `S`, over every pooled path:

- **`pairs_outside_before_set`** *(this is what the greedy minimises)* — for each set
  letter α, the number of outside letters (β∉S) that first appeared **strictly
  earlier**; a β entering on the **same guess** as α is tied and does **not** count.
  **And if α never appears on that path at all, every outside letter that does appear
  counts against it** — the never-guessed penalty, so rare letters are penalised rather
  than rewarded for their absence. Summed over α∈S.
- **`set_before_outside`** *(reported alongside, does not steer the greedy)* — the
  number of set-letter **occurrences in the answer** that are revealed before the first
  outside letter is typed. A set letter entering on the same guess as that first outside
  letter is tied, so it does not count as before.

Both are averaged **uniformly over the pooled paths** (the Wordle sweep carries no
frequency weighting), so with `NUM_TREES = 1` that is a plain average over the 2315
answers.

In [ ]:

def _in_set(letters):
    """A 27-slot boolean mask over letter indices; slot 26 is the padding sentinel."""
    inS = np.zeros(27, dtype=bool)
    for c in str(letters).lower():
        if c.isalpha():
            inS[ord(c) - 97] = True
    return inS


def pairs_total(inS, D=D):
    """Total outside-before-set pairs over every pooled path (see the markdown above)."""
    valid = D['valid']
    inS_seq = inS[D['order']] & valid
    out_seq = valid & ~inS_seq
    cum = np.cumsum(out_seq, axis=1) - out_seq              # outside letters in EARLIER columns
    before = np.take_along_axis(cum, D['block'], axis=1)    # ... rewound to the tie group's start,
                                                            #     so same-guess letters never count
    old = (inS_seq * before).sum(axis=1)                    # pairs for set letters that DO appear
    n_out = out_seq.sum(axis=1)                             # outside letters appearing on this path
    k_missing = int(inS[:26].sum()) - inS_seq.sum(axis=1)   # set letters that never appear
    return float((old + k_missing * n_out).sum())


def before_total(inS, D=D):
    """Total set-letter occurrences in the answer revealed before the first outside letter."""
    valid = D['valid']
    inS_seq = inS[D['order']] & valid
    out_seq = valid & ~inS_seq
    first_out = np.where(out_seq, D['rank'], BIG).min(axis=1)   # rank of the first outside letter
    mask = inS_seq & (D['rank'] < first_out[:, None])           # strictly earlier -> ties excluded
    return float((D['occ'] * mask).sum())


def letter_set_stats(letters, D=D):
    """Both metrics for one explicit set, averaged over the pooled paths."""
    inS = _in_set(letters)
    P = D['n']
    return {'set': ''.join(sorted(chr(97 + i) for i in range(26) if inS[i])),
            'set_before_outside': before_total(inS, D) / P,
            'pairs_outside_before_set': pairs_total(inS, D) / P}


# Sanity: the full alphabet has no outside letters, so pairs must be 0 and every one of
# the answer's five letter-occurrences is revealed before a (non-existent) outside letter.
display(pd.DataFrame([letter_set_stats('abcdefghijklmnopqrstuvwxyz'),
                      letter_set_stats('salet'),
                      letter_set_stats('aeiou')]))


## Greedily grow the set to minimise outside-before-set pairs

Starting from the empty set, repeatedly add the outside letter that **minimises
`pairs_outside_before_set`** (with the never-guessed rule, so rare letters are not
favoured), until all 26 are in. Ties are broken alphabetically.

Each step prints the added letter, the resulting set, that set's `set_before_outside`
and `pairs_outside_before_set`, and **`top2_diff`** — the runner-up candidate's pairs
minus the chosen letter's, i.e. how decisively the winner beat the next-best option
(NaN at the final step). A blank line marks `top2_diff > MIN_GAP`.

Because a path's letter order does not depend on the set, the orders are collected once
(above) and each candidate set is scored by a vectorised pass over the padded arrays.

In [ ]:

def greedy_min_pairs(D=D, min_gap=None):
    """Greedily add one letter at a time -- each step the outside letter that MINIMISES
    pairs_outside_before_set -- until all 26 are in. Returns the df."""
    if min_gap is None:
        min_gap = MIN_GAP
    P = D['n']
    inS = np.zeros(27, dtype=bool)
    rows, chosen, remaining = [], [], list(range(26))

    names = ', '.join(w for w, _t in D['openers'][:4])
    more = '' if len(D['openers']) <= 4 else f' (+{len(D["openers"]) - 4} more)'
    print(f'greedy min pairs_outside_before_set  |  {len(D["openers"])} tree(s): {names}{more}'
          f'  |  {P:,} paths, uniform')

    for step in range(26):
        scored = []
        for x in remaining:                      # pairs total for each outside candidate
            inS[x] = True
            scored.append((pairs_total(inS, D), x))
            inS[x] = False
        scored.sort()                            # (pairs, letter) asc -> ties alphabetical
        best_tot, best_x = scored[0]
        second_tot = scored[1][0] if len(scored) > 1 else None

        inS[best_x] = True
        remaining.remove(best_x)
        chosen.append(best_x)

        p_avg = best_tot / P
        gap = (second_tot - best_tot) / P if second_tot is not None else np.nan
        b_avg = before_total(inS, D) / P
        letter = chr(97 + best_x)
        setstr = ''.join(sorted(chr(97 + c) for c in chosen))
        rows.append({'step': step + 1, 'added': letter, 'set': setstr,
                     'set_before_outside': round(b_avg, 4),
                     'pairs_outside_before_set': round(p_avg, 4),
                     'top2_diff': round(gap, 4) if second_tot is not None else np.nan})
        print(f'  step {step + 1:2d}: +{letter}   set={setstr:<26}  '
              f'set_before_outside={b_avg:7.4f}   pairs={p_avg:7.4f}   top2_diff={gap:7.4f}')
        if second_tot is not None and gap > min_gap:
            print()
    return pd.DataFrame(rows)


sel = greedy_min_pairs()
display(sel)


## The resulting order

`ORDER_LETTERS` is the greedy ranking, best first. **The order is computed; the tier
CUTS are an editorial decision** — in hangman the `top2_diff` gaps informed where the
rows were drawn, but the final seven tiers merged some of those groups by hand before
being copied into `animations/assets/tier_list.py`. Same convention here: read the gaps,
then choose the cuts deliberately.

In [ ]:

ORDER_LETTERS = ''.join(sel['added'])
print(f'order ({len(D["openers"])} tree(s)): {ORDER_LETTERS.upper()}')

gaps = sel[sel['top2_diff'] > MIN_GAP][['step', 'added', 'top2_diff']]
print(f'\nsteps where top2_diff > {MIN_GAP} (candidate tier boundaries, cut AFTER the step):')
display(gaps)